<a href="https://colab.research.google.com/github/minurikahangama/Automation-test-for-the-inventory-system/blob/main/phps_training_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Updated Cell 1: Install compatible library pairs
!pip install --upgrade transformers accelerate datasets scikit-learn pandas -q

import os
print("✓ Ecosystem updated. Restarting memory kernel to apply fixes...")
os.kill(os.getpid(), 9)

In [ ]:
import pandas as pd
df = pd.read_csv('phps_training_final.csv')
print("✓ Dataset loaded successfully")
print(f"Total rows: {len(df)}")
print(f"Label counts:\n{df['label'].value_counts()}")

✓ Dataset loaded successfully
Total rows: 450
Label counts:
label
2    150
0    150
1    150
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)
print(f"✓ Training set: {len(train_df)} rows")
print(f"✓ Test set:     {len(test_df)} rows")

✓ Training set: 360 rows
✓ Test set:     90 rows


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

# Fixed: Updated to the correct, accessible model identity string
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True)).map(tokenize, batched=True).rename_column("label", "labels")
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True)).map(tokenize, batched=True).rename_column("label", "labels")

train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_ds.set_format('torch',  columns=['input_ids', 'attention_mask', 'labels'])
print("✓ Tokenisation complete")

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Map:   0%|          | 0/360 [00:00<?, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

✓ Tokenisation complete


In [ ]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True
)
print("✓ Model loaded successfully (0=Negative, 1=Neutral, 2=Positive)")

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model loaded successfully (0=Negative, 1=Neutral, 2=Positive)


In [ ]:
import numpy as np
from sklearn.metrics import f1_score
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {'f1': f1_score(labels, predictions, average='weighted')}

training_args = TrainingArguments(
    output_dir                  = './roberta_phps',
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    warmup_steps                = 50,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',  # Fixed: Changed from evaluation_strategy
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    fp16                        = True,
    logging_steps               = 20,
    report_to                   = 'none',
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = test_ds,
    compute_metrics = compute_metrics,
)
print("✓ Training setup ready. Run the next cell to begin.")

✓ Training setup ready. Run the next cell to begin.


In [ ]:
print("Core Fine-Tuning execution initiated... This takes roughly 5-10 minutes.")
trainer.train()
print("✓ Training complete!")

Core Fine-Tuning execution initiated... This takes roughly 5-10 minutes.


Epoch,Training Loss,Validation Loss,F1
1,0.448944,0.448907,0.821057
2,0.327738,0.534199,0.821805
3,0.138619,0.727403,0.831797


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Training complete!


In [ ]:
from sklearn.metrics import classification_report

preds_output = trainer.predict(test_ds)
predictions  = np.argmax(preds_output.predictions, axis=-1)
true_labels  = test_df['label'].values

print("=" * 50)
print("             PHPS VERIFICATION REPORT           ")
print("=" * 50)
print(classification_report(true_labels, predictions, target_names=['Negative', 'Neutral', 'Positive']))

f1 = f1_score(true_labels, predictions, average='weighted')
if f1 >= 0.80:
    print(f"✓ NFR-04 PASSED — F1 Score: {f1:.4f} achieved!")
else:
    print(f"⚠ Target under F1 0.80 (Achieved: {f1:.4f}). Safe for BSc thesis milestones, but add more data lines to scale accuracy.")

             PHPS VERIFICATION REPORT           
              precision    recall  f1-score   support

    Negative       0.75      0.80      0.77        30
     Neutral       0.85      0.73      0.79        30
    Positive       0.91      0.97      0.94        30

    accuracy                           0.83        90
   macro avg       0.83      0.83      0.83        90
weighted avg       0.83      0.83      0.83        90

✓ NFR-04 PASSED — F1 Score: 0.8318 achieved!


In [ ]:
trainer.save_model('./roberta_phps')
tokenizer.save_pretrained('./roberta_phps')
print("✓ Weights and configurations saved to directory cache.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Weights and configurations saved to directory cache.


In [ ]:
import shutil
shutil.make_archive('roberta_phps', 'zip', '.', 'roberta_phps')
print("✓ Packaged archive created. Right-click 'roberta_phps.zip' in your file menu to download.")

✓ Packaged archive created. Right-click 'roberta_phps.zip' in your file menu to download.


In [ ]:
import os
import shutil

# 1. Clean up any corrupted or looping zip files from earlier
if os.path.exists('roberta_phps.zip'):
    os.remove('roberta_phps.zip')

# 2. Target ONLY the specific folder containing your model weights
# This explicitly prevents the infinite zip loop bug
shutil.make_archive(base_name='phps_model_fixed', format='zip', root_dir='/content/roberta_phps')

# 3. Check the exact file size to ensure it is healthy
size_bytes = os.path.getsize('phps_model_fixed.zip')
size_mb = size_bytes / (1024 * 1024)
print(f"✓ Success! Clean zip created.")
print(f"✓ Exact File Size: {size_mb:.2f} MB (Should be around 400-500 MB)")

✓ Success! Clean zip created.
✓ Exact File Size: 3612.66 MB (Should be around 400-500 MB)
